# **DATA INFORMATION QUALITY PROJECT**
----

**Stefan Calugaru**  
Personal Code: 10852010 – Student ID: 212886  

**Stefano Molteni**  
Personal Code: 10894104 – Student ID: 306650  

**Gabriele Pedesini**  
Personal Code: 10840961 – Student ID: 304425

In [1]:
# import necessary libraries
import pandas as pd
import numpy as np
from datetime import datetime

In [11]:
# import dataset
DATASET = pd.read_csv('Comune-di-Milano-Pubblici-esercizi.csv', sep=';', encoding='utf-16')


# fix unrecognized character issues
replacements = {
    r'CAFF\uFFFD': 'CAFFÈ',
    r'caff\uFFFD': 'caffè',
    r'([0-9])\uFFFD(\s?ingr)': r'\1°\2',                   # 2°ingresso / 2°ingr
    r'([0-9])\uFFFD(\s?piano|\s?p.interrato)': r'\1°\2',   # 8°piano / 2°p.interrato
    r'(n)\uFFFD(\s?[0-9])': r'\1°\2'                       # via trenno n°20
}

str_columns = DATASET.select_dtypes(include='object').columns

for col in str_columns:
    for pattern, repl in replacements.items():
        DATASET[col] = DATASET[col].str.replace(pattern, repl, regex=True)

DATASET

,Tipo esercizio storico pe,Insegna,Ubicazione,Tipo via,Descrizione via,Civico,Codice via,ZD,Forma commercio,Forma commercio prev,Forma vendita,Settore storico pe,Superficie somministrazione
0,NaN,NaN,ALZ NAVIGLIO GRANDE N. 12 ; isolato:057; (z.d. 6),ALZ,NAVIGLIO GRANDE,12,5144,6,NaN,NaN,NaN,"Ristorante, trattoria, osteria;Genere Merceol....",83.0
1,NaN,NaN,ALZ NAVIGLIO GRANDE N. 44 (z.d. 6),ALZ,NAVIGLIO GRANDE,44,5144,6,NaN,NaN,NaN,Bar gastronomici e simili,26.0
2,NaN,NaN,ALZ NAVIGLIO GRANDE N. 48 (z.d. 6),ALZ,NAVIGLIO GRANDE,48,5144,6,NaN,NaN,NaN,Bar gastronomici e simili,58.0
3,NaN,NaN,ALZ NAVIGLIO GRANDE N. 8 (z.d. 6),ALZ,NAVIGLIO GRANDE,8,5144,6,NaN,NaN,NaN,"BAR CAFFÈ E SIMILI;Ristorante, trattoria, osteria",101.0
4,NaN,NaN,ALZ NAVIGLIO PAVESE N. 24 (z.d. 6),ALZ,NAVIGLIO PAVESE,24,5161,6,NaN,NaN,NaN,Bar gastronomici e simili,51.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6899,"wine,birr.,pub enot.,caff.,the",bar cherry,VLE DORIA ANDREA N. 12 ; isolato:031; accesso:...,VLE,DORIA ANDREA,12,2230,2,solo somministrazione,somministrazione,misto,"Wine,birr.,pub enot.,caff.,the",59.0
6900,"wine,birr.,pub enot.,caff.,the",la balusa,VIA GARIGLIANO N. 5 ; isolato:277; accesso: ac...,VIA,GARIGLIANO,5,1134,9,solo somministrazione,somministrazione,misto,"Wine,birr.,pub enot.,caff.,the",40.0
6901,"wine,birr.,pub enot.,caff.,the",la champagnerie sas,VIA SOTTOCORNO PASQUALE N. 4 ; isolato:014; ac...,VIA,SOTTOCORNO PASQUALE,4,3152,4,solo somministrazione,somministrazione,misto,BAR CAFFÈ E SIMILI;Bar gastronomici e simili,53.0
6902,"wine,birr.,pub enot.,caff.,the",old rooster,VIA CASTROVILLARI N. 23 ; isolato:150; accesso...,VIA,CASTROVILLARI,23,6299,7,solo somministrazione,somministrazione,misto,"Wine,birr.,pub enot.,caff.,the",43.0


## **1. Data Quality Assessment**
----

In [45]:
# general info about the dataset, such as number of non-null entries and data types for each column
DATASET.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6904 entries, 0 to 6903
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Tipo esercizio storico pe    5551 non-null   object 
 1   Insegna                      3494 non-null   object 
 2   Ubicazione                   6904 non-null   object 
 3   Tipo via                     6904 non-null   object 
 4   Descrizione via              6904 non-null   object 
 5   Civico                       6748 non-null   object 
 6   Codice via                   6904 non-null   int64  
 7   ZD                           6904 non-null   int64  
 8   Forma commercio              5333 non-null   object 
 9   Forma commercio prev         5507 non-null   object 
 10  Forma vendita                5480 non-null   object 
 11  Settore storico pe           6883 non-null   object 
 12  Superficie somministrazione  6825 non-null   float64
dtypes: float64(1), int

In [46]:
# descriptive statistics about the numeric columns of the dataset
DATASET.describe()

,Codice via,ZD,Superficie somministrazione
count,6904.000000,6904.000000,6825.000000
mean,3555.016367,4.573001,85.996777
std,2238.982545,2.763538,89.676674
min,1.000000,1.000000,2.000000
25%,1510.000000,2.000000,42.000000
50%,3129.000000,4.000000,64.000000
75%,5294.000000,7.000000,100.000000
max,7602.000000,9.000000,2336.000000


In [47]:
# count of unique values in the "Tipo esercizio storico pe" column
DATASET["Tipo esercizio storico pe"].value_counts()

Tipo esercizio storico pe
bar caffè                         3274
ristorante                         807
trattoria                          554
pizzeria                           350
tavola calda                       104
spaccio bevande analcoliche         82
ristorante, trattoria, osteria      62
bar gastronomici e simili           49
gelateria                           48
osteria                             38
tavola fredda                       34
bar-caffe' e simili                 30
genere merceol.autorizz.sanit.      27
birreria                            21
pizzerie e simili                   16
bar pasticc.gelat.crem.creper.      11
cibi cotti                          10
tav.calde,self service,fast f.      10
cibi cotti preconfezionati           7
prodotti di gastronomia              7
wine,birr.,pub enot.,caff.,the       5
sale da ballo, locali notturni       4
mensa                                1
Name: count, dtype: int64

In [48]:
# count of unique values in the "Settore storico pe" column
DATASET["Settore storico pe"].value_counts()

Settore storico pe
Genere Merceol.Autorizz.Sanit.                                                                                                       214
Ristorante, trattoria, osteria                                                                                                       174
Bar gastronomici e simili                                                                                                            165
BAR CAFFÈ E SIMILI                                                                                                                    85
BAR CAFFÈ E SIMILI;Bar gastronomici e simili                                                                                          55
                                                                                                                                    ... 
Seconda bottiglia;Trattoria;BAR CAFFÈ E SIMILI;Ristorante, trattoria, osteria                                                          1
Biliardo;BAR CAFFÈ;Ris

In [49]:
# count of unique values in the "Forma commercio" column
DATASET["Forma commercio"].value_counts()

Forma commercio
solo somministrazione      4763
somministrazione/minuto     570
Name: count, dtype: int64

In [50]:
# count of unique values in the "Forma commercio prev" column
DATASET["Forma commercio prev"].value_counts()

Forma commercio prev
somministrazione    5387
minuto               120
Name: count, dtype: int64

In [51]:
# count of unique values in the "Forma vendita" column
DATASET["Forma vendita"].value_counts()

Forma vendita
misto           2212
al banco        2133
al tavolo       1087
self service      48
Name: count, dtype: int64

#### **1.1. Duplication**

In [55]:
# identify duplicate rows in the dataset
DUPLICATES = DATASET.duplicated()
DATASET[DUPLICATES]

,Tipo esercizio storico pe,Insegna,Ubicazione,Tipo via,Descrizione via,Civico,Codice via,ZD,Forma commercio,Forma commercio prev,Forma vendita,Settore storico pe,Superficie somministrazione
940,NaN,NaN,VIA SALASCO N. 29 (z.d. 5),VIA,SALASCO,29,4051,5,NaN,NaN,NaN,Spaccio bevande analcoliche,NaN


#### **1.2. Completeness**

In [65]:
# number of non-null entries in the entire dataset
NOT_NULL = DATASET.count().sum()

# total number of entries in the dataset
TOT = DATASET.shape[0] * DATASET.shape[1]

# calculation of completeness metric
COMPLETENESS = NOT_NULL / TOT

COMPLETENESS = '{0:.1f}%'.format(COMPLETENESS*100)
COMPLETENESS


'89.5%'